In [1]:
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sam2.addons import write_h5
from skimage.measure import label

from skimage.measure import regionprops_table
import pandas as pd

In [6]:
root = Path("/Volumes/warm_SD/data/285/00_series/00")

# Get only files directly in root
all_tifs = sorted([f for f in root.iterdir() if f.suffix.lower() in [".tif", ".tiff"]])

images, masks, instances, binaries = [], [], [], []

for f in all_tifs:
    name = f.name.lower()
    if ".mask" in name:
        masks.append(f)
    elif ".instances" in name:
        instances.append(f)
    elif ".bin" in name:
        binaries.append(f)
    else:
        images.append(f)

print(f"Found {len(images)} images, {len(masks)} masks, {len(instances)} instances,{len(binaries)} binaries.")


Found 3 images, 3 masks, 3 instances,3 binaries.


In [7]:

def extract_instances_from_labelmap(labelmap: np.ndarray):
    """
    Extract instance crops and bounding boxes from a labelmap.
    
    Parameters
    ----------
    labelmap : np.ndarray
        2D array of shape (H, W) with integer labels (int16 or int32).
        0 is assumed to be background.
    
    Returns
    -------
    instances : dict
        Dictionary where each key is an instance id (int)
        and each value is a dict containing:
        - 'bbox': [x, y, w, h]
        - 'segmentation_crop': np.ndarray (cropped mask of that instance)
    """
    instances = []
    ids = np.unique(labelmap)
    ids = ids[ids != 0]  # remove background (0)
    print(len(ids))
    for inst_id in ids:
        mask = labelmap == inst_id
        ys, xs = np.where(mask)

        if len(xs) == 0 or len(ys) == 0:
            continue

        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        # width and height (inclusive bounding box)
        w = x_max - x_min + 1
        h = y_max - y_min + 1

        # cropped binary mask
        segmentation_crop = mask[y_min:y_max+1, x_min:x_max+1].astype(np.bool_)

        instances.append({
            "bbox": [int(x_min), int(y_min), int(w), int(h)],
            "segmentation_crop": segmentation_crop,
            "id":inst_id
        })
        # if inst_id == 4:
        #     break
    
    return instances

In [8]:
for image_path, mask_path, instance_path, binary_path in zip(images[:], masks[:], instances[:], binaries[:]):
    # ---- check names ----
    img_name = image_path.stem
    mask_name = mask_path.stem.replace(".mask", "")
    instance_name = instance_path.stem.replace(".instances", "")
    bin_name = binary_path.stem.replace(".bin", "")
    
    assert img_name == mask_name == instance_name == bin_name, \
        f"File name mismatch:\n  {image_path.name}\n  {mask_path.name}\n  {instance_path.name} \n{binary_path.name}"

    print(f"✅ {img_name}")

    # ---- load files ----
    image = np.array(Image.open(image_path), dtype=np.uint8)       # grayscale or RGB
    mask = np.array(Image.open(mask_path), dtype=bool)             # binary mask
    labels = np.array(Image.open(instance_path), dtype=np.int16) # or np.int16 if sufficient
    bin = np.array(Image.open(binary_path), dtype=bool) # or np.int16 if sufficient
    
    if len(np.unique(labels)) <= 2:
        labels = label(labels)


    # # multiply with mask but keep dtype
    # labels_masked = (labels * mask).astype(labels.dtype)

    # props = regionprops_table(
    #     labels_masked,
    #     properties=('centroid', 'equivalent_diameter', 'axis_major_length', 'axis_minor_length'),
    # )

    # df_props = pd.DataFrame(props)

    # df_props = df_props.sort_values(by="equivalent_diameter", ascending=True).reset_index(drop=True)



    instances_ = extract_instances_from_labelmap(labels)

    # ---- optional: verify shapes ----
    assert image.shape[:2] == mask.shape[:2] == labels.shape[:2], \
        f"Shape mismatch in {img_name}: {image.shape}, {mask.shape}, {labels.shape}"

    out_dict = {"image": image,
                "labels": labels,
                "mask": mask,
                "binary": bin,
                "instances": instances_}
    
    write_h5(path=f"{img_name}.h5",dict_out=out_dict)

   


✅ 285_00_01_50x
3630
✅ Saved: 285_00_01_50x.h5
  ├─ image shape   : (2636, 16140)
  ├─ labels shape  : (2636, 16140)
  ├─ binary shape  : (2636, 16140)
  ├─ mask shape    : (2636, 16140)
  └─ instances     : 3630
✅ 285_00_02_50x
3200
✅ Saved: 285_00_02_50x.h5
  ├─ image shape   : (2687, 17260)
  ├─ labels shape  : (2687, 17260)
  ├─ binary shape  : (2687, 17260)
  ├─ mask shape    : (2687, 17260)
  └─ instances     : 3200
✅ 285_00_03_50x
3455
✅ Saved: 285_00_03_50x.h5
  ├─ image shape   : (2874, 16515)
  ├─ labels shape  : (2874, 16515)
  ├─ binary shape  : (2874, 16515)
  ├─ mask shape    : (2874, 16515)
  └─ instances     : 3455
